# 01 — 数据清洗 & 预处理

**目标**: 将原始交易数据清洗为标准分析数据集，为后续 EDA 和 RFM 建模打好基础。

**步骤**:
1. 加载数据 → 了解数据结构
2. 检查缺失值、重复值、异常值
3. 修正数据类型
4. 创建派生特征（年/月/季度/处理天数）
5. 导出清洗后数据

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示（Windows 可能需要调整）
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('Libraries loaded successfully.')

In [ ]:
# 加载数据
df = pd.read_csv('../data/Sample_Superstore.csv')
print(f'Shape: {df.shape}')
print(f'Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB')

In [ ]:
# ========== 1. 数据结构概览 ==========
print('=== 数据类型 ===')
print(df.dtypes)
print(f'\n=== 前5行 ===')
df.head()

In [ ]:
# ========== 2. 缺失值检查 ==========
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Pct %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_df) > 0:
    print('⚠️  发现缺失值:')
    print(missing_df)
else:
    print('✅ 没有缺失值')

In [ ]:
# ========== 3. 重复值检查 ==========
duplicates = df.duplicated().sum()
print(f'完全重复行: {duplicates}')

# 检查 Row ID 是否唯一
row_id_dupes = df['Row ID'].duplicated().sum()
print(f'Row ID 重复: {row_id_dupes}')

# 检查 Order ID + Product ID 组合
order_product_dupes = df.duplicated(subset=['Order ID', 'Product ID']).sum()
print(f'同一订单同一产品重复: {order_product_dupes}')

In [ ]:
# ========== 4. 基本统计量 ==========
print('=== 数值列描述统计 ===')
df[['Sales', 'Quantity', 'Discount', 'Profit']].describe()

In [ ]:
# ========== 5. 业务逻辑检查 ==========

# 5a. Sales 应为正数
negative_sales = df[df['Sales'] <= 0]
print(f'Sales <= 0: {len(negative_sales)} 行')

# 5b. Quantity 应 >= 1
zero_qty = df[df['Quantity'] < 1]
print(f'Quantity < 1: {len(zero_qty)} 行')

# 5c. Discount 应在 0-1 之间
invalid_discount = df[(df['Discount'] < 0) | (df['Discount'] > 1)]
print(f'Discount 超出 [0,1]: {len(invalid_discount)} 行')

# 5d. Ship Date 应 >= Order Date
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
bad_ship = df[df['Ship Date'] < df['Order Date']]
print(f'发货日期早于下单日期: {len(bad_ship)} 行')

# 5e. Profit 可能为负，但检查极端异常值
q1, q3 = df['Profit'].quantile([0.01, 0.99]).values
extreme_profit = df[(df['Profit'] < q1 * 3) | (df['Profit'] > q3 * 3)]
print(f'极端 Profit 异常值: {len(extreme_profit)} 行')

In [ ]:
# ========== 6. 类别变量检查 ==========
categorical_cols = ['Ship Mode', 'Segment', 'Country', 'Region', 'Category', 'Sub-Category']

for col in categorical_cols:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().to_string())
    print(f'Unique values: {df[col].nunique()}')

In [ ]:
# ========== 7. 创建派生特征 ==========

# 时间维度
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Quarter'] = df['Order Date'].dt.quarter
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')
df['Order Day of Week'] = df['Order Date'].dt.dayofweek
df['Order Day Name'] = df['Order Date'].dt.day_name()

# 处理时长（天）
df['Processing Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# 盈利标记
df['Is Profitable'] = df['Profit'] > 0

# 折扣分桶
df['Discount Tier'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.1, 0.3, 1.0],
    labels=['No Discount', 'Low (0-10%)', 'Medium (10-30%)', 'High (>30%)']
)

# 利润率
df['Profit Margin'] = (df['Profit'] / df['Sales'] * 100).round(2)

print('✅ 派生特征已创建')
print(f'新增列: Order Year, Order Month, Order Quarter, Processing Days, Is Profitable, Discount Tier, Profit Margin')

In [ ]:
# ========== 8. 最终数据质量报告 ==========
print('======== 数据质量报告 ========')
print(f'总行数: {len(df):,}')
print(f'总列数: {len(df.columns)}')
print(f'订单数: {df["Order ID"].nunique():,}')
print(f'客户数: {df["Customer ID"].nunique():,}')
print(f'产品数: {df["Product ID"].nunique():,}')
print(f'日期范围: {df["Order Date"].min().date()} ~ {df["Order Date"].max().date()}')
print(f'总销售额: ${df["Sales"].sum():,.0f}')
print(f'总利润: ${df["Profit"].sum():,.0f}')
print(f'整体利润率: {df["Profit"].sum() / df["Sales"].sum() * 100:.2f}%')
print(f'平均处理天数: {df["Processing Days"].mean():.1f} 天')
print(f'亏损交易占比: {(~df["Is Profitable"]).mean() * 100:.2f}%')
print('=' * 35)

In [ ]:
# ========== 9. 保存清洗后数据 ==========
cleaned_path = '../data/Superstore_Cleaned.csv'
df.to_csv(cleaned_path, index=False)
print(f'✅ Cleaned data saved to: {cleaned_path}')
print(f'Columns: {list(df.columns)}')

---
## 清洗总结

| 检查项 | 结果 |
|--------|------|
| 缺失值 | ✅ 无缺失 |
| 重复值 | ✅ 无重复 |
| 数据类型 | ✅ 日期列已转为 datetime |
| 业务逻辑 | ✅ Sales/Quantity/Discount 均在合理范围 |
| 派生特征 | ✅ 新增 9 个分析用列 |

**下一步**: → `02_eda.ipynb` 探索性数据分析